# HAM10000 — Clasificación con fuentes sintéticas aisladas

Evalúa el efecto de cada fuente sintética por separado, controlando la cantidad (N=801 en escenarios `_2x`).

| Escenario | Train mel | Fuente sintética | Pregunta |
|---|---|---|---|
| `img2img_2x` | 801 real + 801 img2img | img2img | ¿img2img es mejor que mezcla? |
| `ti_filtered_2x` | 801 real + 801 TI filtradas | TI (solo calidad OK) | ¿Filtrar TI mejora? |
| `gan64_2x` | 801 real + 801 GAN-64px (old) | WGAN-GP 64×64 — 1000 imgs | Baseline GAN |
| `gan_final_2x` | 801 real + 801 GAN-final | WGAN-GP 64×64 — 5000 imgs (PIL) | ¿Más imágenes GAN mejoran? |
| `lora_2x` | 801 real + 801 LoRA | DreamBooth-LoRA rank=32 | ¿LoRA supera TI? |
| `img2img_all` | 801 real + 2403 img2img | img2img (todo) | ¿Más img2img ayuda? |
| `img2img_only` | 0 real + 801 img2img | img2img | ¿img2img reemplaza reales? |
| `derm_s040_2x` | 801 real + 801 Derm-T2IM (s=0.40) | Derm-T2IM — mismo dominio | ¿Menor shift con modelo dermoscopy-specific? |
| `derm_s005_2x` | 801 real + 801 Derm-T2IM (s=0.05) | Derm-T2IM — perturbaciones sutiles | ¿Augmentation conservadora mejora recall? |

Los escenarios `_2x` usan la misma N — solo cambia la fuente. Permite aislar el efecto de la fuente.

**Prerequisitos Colab:**
- `classification_data.zip` en Drive: `Mi unidad/ham10000-augmentation/`
- Carpetas sintéticas en Drive: `synthetic/textual_inversion/`, `synthetic/img2img/`, `synthetic/lora/`, `synthetic/derm_s040/`, `synthetic/derm_s005/`
- ZIPs GAN en Drive: `gan_generated.zip` (old) y `gan_final.zip` (nuevo)

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'CUDA GPU: {name}  ({vram:.1f} GB)')
        return torch.device('cuda'), vram
    if torch.backends.mps.is_available():
        print('Apple Silicon MPS')
        return torch.device('mps'), 0
    print('Sin GPU — usando CPU')
    return torch.device('cpu'), 0

DEVICE, VRAM_GB = get_device()
print(f'IN_COLAB={IN_COLAB}  device={DEVICE}')

In [ ]:
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT     = Path('/content/drive/MyDrive/ham10000-augmentation')
    ZIP_PATH       = DRIVE_ROOT / 'classification_data.zip'
    IMAGES_DIR     = Path('/content/images')
    SPLITS_DIR     = Path('/content/splits')
    SYNTH_ROOT     = DRIVE_ROOT / 'synthetic'
    GAN_ZIP        = next(DRIVE_ROOT.glob('gan_generated*.zip'), None)
    GAN_FINAL_ZIP  = next(DRIVE_ROOT.glob('gan_final*.zip'), None)
    LORA_ZIP       = next(DRIVE_ROOT.glob('lora*.zip'), None)
    GAN_64_DIR     = Path('/content/gan_64')
    GAN_FINAL_DIR  = Path('/content/gan_final')
    EXP_ROOT       = DRIVE_ROOT / 'experiments'
else:
    PROJECT_ROOT   = Path.cwd()
    ZIP_PATH       = PROJECT_ROOT / 'data/processed/classification_data.zip'
    IMAGES_DIR     = PROJECT_ROOT / 'data/processed/images'
    SPLITS_DIR     = PROJECT_ROOT / 'data/processed/splits'
    SYNTH_ROOT     = PROJECT_ROOT / 'data/synthetic'
    GAN_ZIP        = next(PROJECT_ROOT.glob('gan_generated*.zip'), None)
    GAN_FINAL_ZIP  = next(PROJECT_ROOT.glob('gan_final*.zip'), None)
    LORA_ZIP       = next(PROJECT_ROOT.glob('lora*.zip'), None)
    GAN_64_DIR     = PROJECT_ROOT / 'data/synthetic/gan_64'
    GAN_FINAL_DIR  = PROJECT_ROOT / 'data/synthetic/gan_final'
    EXP_ROOT       = PROJECT_ROOT / 'experiments'

LORA_DIR      = SYNTH_ROOT / 'lora'
DERM_S040_DIR = SYNTH_ROOT / 'derm_s040'
DERM_S005_DIR = SYNTH_ROOT / 'derm_s005'
EXP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Imágenes:     {IMAGES_DIR}')
print(f'Sintéticas:   {SYNTH_ROOT}')
print(f'GAN-64 dir:   {GAN_64_DIR}')
print(f'GAN-final dir:{GAN_FINAL_DIR}')
print(f'LoRA dir:     {LORA_DIR}')
print(f'Derm-s040:    {DERM_S040_DIR}')
print(f'Derm-s005:    {DERM_S005_DIR}')
print(f'Experimentos: {EXP_ROOT}')

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import timm; print(f'timm {timm.__version__}')
except ImportError:
    pip_install('timm'); import timm

try:
    import sklearn; print(f'scikit-learn {sklearn.__version__}')
except ImportError:
    pip_install('scikit-learn'); import sklearn

try:
    import cv2; print(f'cv2 {cv2.__version__}')
except ImportError:
    pip_install('opencv-python-headless'); import cv2

print('Dependencias listas')

In [ ]:
# Extraer ZIP de imágenes reales (solo Colab, idempotente)
import zipfile

if IN_COLAB:
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    jpg_count = len(list(IMAGES_DIR.glob('*.jpg')))
    if jpg_count < 100:
        assert ZIP_PATH.exists(), f'ZIP no encontrado: {ZIP_PATH}'
        print('Extrayendo imágenes reales del ZIP...')
        with zipfile.ZipFile(ZIP_PATH) as zf:
            for m in zf.infolist():
                data = zf.read(m.filename)
                if m.filename.startswith('images/') and m.filename.endswith('.jpg'):
                    (IMAGES_DIR / Path(m.filename).name).write_bytes(data)
                elif m.filename.startswith('splits/') and m.filename.endswith('.csv'):
                    (SPLITS_DIR / Path(m.filename).name).write_bytes(data)
        print('Listo')
    else:
        print(f'Imágenes ya extraídas ({jpg_count} archivos)')

In [ ]:
# Copiar sintéticas de Drive a disco local (solo Colab, una vez por sesión)
import shutil

if IN_COLAB:
    LOCAL_SYNTH = Path('/content/synthetic')
    if not LOCAL_SYNTH.exists():
        print('Copiando sintéticas de Drive a /content/synthetic/ ...')
        shutil.copytree(str(SYNTH_ROOT), str(LOCAL_SYNTH))
        print(f'Listo: {len(list(LOCAL_SYNTH.glob("**/*.jpg")))} imágenes')
    else:
        print(f'Sintéticas ya en local: {len(list(LOCAL_SYNTH.glob("**/*.jpg")))} imágenes')
    SYNTH_ROOT    = LOCAL_SYNTH
    LORA_DIR      = SYNTH_ROOT / 'lora'
    DERM_S040_DIR = SYNTH_ROOT / 'derm_s040'
    DERM_S005_DIR = SYNTH_ROOT / 'derm_s005'

n_lora = len(list(LORA_DIR.glob('*.jpg')))          if LORA_DIR.exists()      else 0
n_d040 = len(list(DERM_S040_DIR.glob('*.jpg')))     if DERM_S040_DIR.exists() else 0
n_d005 = len(list(DERM_S005_DIR.glob('*.jpg')))     if DERM_S005_DIR.exists() else 0
print(f'LoRA: {n_lora}  |  Derm-s040: {n_d040}  |  Derm-s005: {n_d005}')

In [ ]:
# Extraer imágenes GAN del ZIP (solo Colab, idempotente)
import struct

if IN_COLAB:
    # GAN original (gan_64)
    GAN_64_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(GAN_64_DIR.glob('*.png'))) < 10 and GAN_ZIP and GAN_ZIP.exists():
        print(f'Extrayendo {GAN_ZIP.name}...')
        with zipfile.ZipFile(GAN_ZIP) as zf:
            for member in zf.infolist():
                if not member.filename.endswith('.png'):
                    continue
                data = zf.read(member.filename)
                w = struct.unpack('>I', data[16:20])[0]
                h = struct.unpack('>I', data[20:24])[0]
                if w == 64 and h == 64:
                    (GAN_64_DIR / Path(member.filename).name).write_bytes(data)
        print(f'  → {len(list(GAN_64_DIR.glob("*.png")))} imágenes GAN-64')

    # GAN final (gan_final) — guardado correctamente con PIL
    GAN_FINAL_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(GAN_FINAL_DIR.glob('*.png'))) < 10:
        assert GAN_FINAL_ZIP and GAN_FINAL_ZIP.exists(), (
            'gan_final.zip no encontrado en Drive.\n'
            'Sube el archivo como: Mi unidad/ham10000-augmentation/gan_final.zip'
        )
        print(f'Extrayendo {GAN_FINAL_ZIP.name}...')
        with zipfile.ZipFile(GAN_FINAL_ZIP) as zf:
            for m in zf.infolist():
                if m.filename.endswith('.png'):
                    (GAN_FINAL_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))
        print(f'  → {len(list(GAN_FINAL_DIR.glob("*.png")))} imágenes GAN final')

    # LoRA — extraer si no está en synthetic/ ya
    if not LORA_DIR.exists() or len(list(LORA_DIR.glob('*.jpg'))) < 10:
        if LORA_ZIP and LORA_ZIP.exists():
            LORA_DIR.mkdir(parents=True, exist_ok=True)
            print(f'Extrayendo {LORA_ZIP.name}...')
            with zipfile.ZipFile(LORA_ZIP) as zf:
                for m in zf.infolist():
                    if m.filename.lower().endswith('.jpg'):
                        (LORA_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))
            print(f'  → {len(list(LORA_DIR.glob("*.jpg")))} imágenes LoRA')

n64    = len(list(GAN_64_DIR.glob('*.png'))) if GAN_64_DIR.exists() else 0
nfinal = len(list(GAN_FINAL_DIR.glob('*.png'))) if GAN_FINAL_DIR.exists() else 0
nlora  = len(list(LORA_DIR.glob('*.jpg'))) if LORA_DIR.exists() else 0
print(f'GAN-64: {n64}  |  GAN-final: {nfinal}  |  LoRA: {nlora}')


## Estado de escenarios
Ejecuta esta celda para ver el progreso sin cargar modelos.

In [ ]:
import json

SCENARIOS = ['img2img_2x', 'ti_filtered_2x', 'gan64_2x', 'gan_final_2x', 'lora_2x',
             'img2img_all', 'img2img_only', 'derm_s040_2x', 'derm_s005_2x']

def find_run_dir(sc):
    pointer = EXP_ROOT / f'{sc}_current.txt'
    if pointer.exists():
        return EXP_ROOT / pointer.read_text().strip()
    completed = sorted([d for d in EXP_ROOT.glob(f'*_{sc}') if (d / 'test_metrics.json').exists()])
    return completed[-1] if completed else None

print('Estado de escenarios:')
for sc in SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir is None:
        print(f'  {sc}: ⬜ pendiente')
    elif (run_dir / 'test_metrics.json').exists():
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f'  {sc}: ✅ ({run_dir.name})  AUC={m["auc"]}  Recall={m["recall_mel"]}  F1={m["f1_mel"]}')
    else:
        ckpt   = run_dir / 'checkpoint_last.pt'
        status = '🔄 retomable' if ckpt.exists() else '⏳ iniciado'
        print(f'  {sc}: {status} ({run_dir.name})')

ti_paths     = list((SYNTH_ROOT / 'textual_inversion').glob('*.jpg')) if (SYNTH_ROOT / 'textual_inversion').exists() else []
i2i_paths    = list((SYNTH_ROOT / 'img2img').glob('*.jpg')) if (SYNTH_ROOT / 'img2img').exists() else []
gan_paths    = list(GAN_64_DIR.glob('*.png')) if GAN_64_DIR.exists() else []
gfinal_paths = list(GAN_FINAL_DIR.glob('*.png')) if GAN_FINAL_DIR.exists() else []
lora_paths   = list(LORA_DIR.glob('*.jpg')) if LORA_DIR.exists() else []
d040_paths   = list(DERM_S040_DIR.glob('*.jpg')) if DERM_S040_DIR.exists() else []
d005_paths   = list(DERM_S005_DIR.glob('*.jpg')) if DERM_S005_DIR.exists() else []
print(f'\nFuentes — TI:{len(ti_paths)} img2img:{len(i2i_paths)} GAN-64:{len(gan_paths)} '
      f'GAN-final:{len(gfinal_paths)} LoRA:{len(lora_paths)} '
      f'Derm-s040:{len(d040_paths)} Derm-s005:{len(d005_paths)}')

## Hiperparámetros

In [ ]:
EPOCHS = 15
LR     = 1e-4
SEED   = 42

if DEVICE.type == 'cuda':
    BATCH_SIZE  = 64 if VRAM_GB >= 40 else 32
    NUM_WORKERS = 2
else:
    BATCH_SIZE  = 8
    NUM_WORKERS = 0

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'EPOCHS={EPOCHS}  LR={LR}  BATCH_SIZE={BATCH_SIZE}  NUM_WORKERS={NUM_WORKERS}')

## Utilidades de dataset y filtro de calidad

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class FlatDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def class_weights(self):
        labels  = np.array([l for _, l in self.samples])
        counts  = np.bincount(labels)
        w_class = 1.0 / counts.astype(float)
        return torch.tensor([w_class[l] for _, l in self.samples], dtype=torch.float)


def make_loader(samples, transform, weighted=False):
    ds  = FlatDataset(samples, transform)
    pin = DEVICE.type == 'cuda'
    if weighted:
        sampler = WeightedRandomSampler(ds.class_weights(), len(ds), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                         num_workers=NUM_WORKERS, pin_memory=pin)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=pin)


def resolve_path(rel_path):
    if IN_COLAB:
        return IMAGES_DIR / Path(rel_path).name
    return Path.cwd() / rel_path


def load_splits():
    def read(csv_path, label_filter=None):
        df = pd.read_csv(csv_path)
        if label_filter is not None:
            df = df[df['label'] == label_filter]
        return [(resolve_path(row['image_path']), int(row['label'])) for _, row in df.iterrows()]

    train_nv  = read(SPLITS_DIR / 'train.csv', label_filter=0)
    train_mel = read(SPLITS_DIR / 'train.csv', label_filter=1)
    val       = read(SPLITS_DIR / 'val.csv')
    test      = read(SPLITS_DIR / 'test.csv')
    print(f'Splits — train nv:{len(train_nv)} mel:{len(train_mel)} | val:{len(val)} | test:{len(test)}')
    return train_nv, train_mel, val, test


# --- Filtro de calidad (mismo criterio que HAM10000_quality_evaluation.ipynb) ---
def passes_quality_filter(img_path: Path) -> bool:
    """True si la imagen pasa el filtro mínimo de calidad visual."""
    try:
        img = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        mean_v   = float(hsv[:, :, 2].mean())
        std_rgb  = float(img.std(axis=(0, 1)).mean())
        h, s, v  = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
        skin_ratio = float(((( h <= 25) | (h >= 150)) & (s > 20) & (v > 20)).mean())
        return (20 < mean_v < 235) and (std_rgb >= 8) and (skin_ratio >= 0.02)
    except Exception:
        return False


def get_filtered_ti(cache_path: Path, seed: int = SEED):
    """Lista reproducible de paths TI que pasan el filtro. Cachea en cache_path."""
    if cache_path.exists():
        paths = [Path(p) for p in json.loads(cache_path.read_text())]
        print(f'TI filtradas (cache): {len(paths)} imágenes')
        return paths
    all_ti = sorted((SYNTH_ROOT / 'textual_inversion').glob('*.jpg'))
    print(f'Aplicando filtro de calidad a {len(all_ti)} imágenes TI...')
    filtered = [p for p in all_ti if passes_quality_filter(p)]
    random.Random(seed).shuffle(filtered)
    cache_path.write_text(json.dumps([str(p) for p in filtered]))
    print(f'TI filtradas: {len(filtered)}/{len(all_ti)} ({100*len(filtered)/len(all_ti):.1f}%)')
    return filtered


def synth_samples_from(paths, n=None, seed=SEED):
    """(Path, 1) list desde una lista de paths, orden reproducible."""
    paths = list(paths)
    random.Random(seed).shuffle(paths)
    if n is not None:
        paths = paths[:n]
    return [(p, 1) for p in paths]


print('Utilidades de dataset listas')

## Loop de entrenamiento con resume

In [ ]:
# Compatibilidad con PyTorch 2.6+ (weights_only=True por defecto rompe timm y numpy)
import numpy as _np
try:
    torch.serialization.add_safe_globals([_np._core.multiarray.scalar])
except AttributeError:
    pass

import time
import timm
import torch.nn as nn
from datetime import datetime, timezone
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt


def build_model():
    return timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)


def _eval_loop(model, loader, criterion):
    model.eval()
    loss_sum, total = 0.0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            loss_sum += loss.item() * len(labels)
            total    += len(labels)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
    return {
        'loss':       round(loss_sum / total, 4),
        'f1_mel':     round(f1_score(all_labels, all_preds, pos_label=1, zero_division=0), 4),
        'recall_mel': round(recall_score(all_labels, all_preds, pos_label=1, zero_division=0), 4),
        'auc':        round(roc_auc_score(all_labels, all_probs), 4),
    }


def run_scenario(name, train_samples, val_samples, test_samples):
    pointer_file = EXP_ROOT / f'{name}_current.txt'
    if pointer_file.exists():
        run_id = pointer_file.read_text().strip()
    else:
        run_id = f'{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}_{name}'
        pointer_file.write_text(run_id)

    run_dir     = EXP_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    done_marker = run_dir / 'test_metrics.json'
    ckpt_path   = run_dir / 'checkpoint_last.pt'

    if done_marker.exists():
        metrics = json.loads(done_marker.read_text())
        pointer_file.unlink(missing_ok=True)
        print(f'[{name}] Ya completado ({run_id})')
        print(f'  AUC={metrics["auc"]}  Recall_mel={metrics["recall_mel"]}  F1_mel={metrics["f1_mel"]}')
        return metrics

    n_nv  = sum(1 for _, l in train_samples if l == 0)
    n_mel = sum(1 for _, l in train_samples if l == 1)
    print(f'\n{"="*60}')
    print(f'Escenario: {name} ({run_id})')
    print(f'train nv={n_nv} mel={n_mel}  |  val={len(val_samples)} test={len(test_samples)}')
    print(f'{"="*60}')

    if not (run_dir / 'config.json').exists():
        config = {
            'run_id': run_id, 'scenario': name, 'model': 'efficientnet_b0',
            'pretrained': True, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
            'lr': LR, 'seed': SEED, 'train_nv': n_nv, 'train_mel': n_mel,
            'started_at': datetime.now(timezone.utc).isoformat(),
        }
        (run_dir / 'config.json').write_text(json.dumps(config, indent=2))

    train_loader = make_loader(train_samples, TRAIN_TF, weighted=True)
    val_loader   = make_loader(val_samples,   EVAL_TF)
    test_loader  = make_loader(test_samples,  EVAL_TF)

    model     = build_model().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    start_epoch = 1
    best_f1     = 0.0
    history     = []

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_f1     = ckpt['best_f1']
        history     = ckpt['history']
        print(f'Retomando desde epoch {start_epoch}  (best_f1={best_f1:.4f})')

    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f'[{name}] Epoch {epoch:02d}/{EPOCHS}', leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
            pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.3f}')

        scheduler.step()
        val_m   = _eval_loop(model, val_loader, criterion)
        elapsed = time.time() - t0

        row = {'epoch': epoch, 'train_loss': round(train_loss/total, 4),
               'train_acc': round(correct/total, 4),
               **{f'val_{k}': v for k, v in val_m.items()},
               'elapsed_s': round(elapsed, 1)}
        history.append(row)
        print(f'  Epoch {epoch:02d}/{EPOCHS} | loss={train_loss/total:.4f} | '
              f'val_f1_mel={val_m["f1_mel"]:.3f} val_auc={val_m["auc"]:.3f} | {elapsed:.0f}s')

        if val_m['f1_mel'] >= best_f1:
            best_f1 = val_m['f1_mel']
            torch.save(model.state_dict(), run_dir / 'best_model.pt')

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
            'best_f1': best_f1, 'history': history,
        }, ckpt_path)

    (run_dir / 'history.json').write_text(json.dumps(history, indent=2))

    model.load_state_dict(torch.load(run_dir / 'best_model.pt', map_location=DEVICE, weights_only=False))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            logits = model(imgs.to(DEVICE))
            probs  = torch.softmax(logits, dim=1)[:, 1]
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())
            all_probs.extend(probs.cpu().tolist())

    report = classification_report(all_labels, all_preds, target_names=['nv', 'mel'], output_dict=True)
    auc    = roc_auc_score(all_labels, all_probs)

    metrics = {
        'auc':           round(auc, 4),
        'f1_mel':        round(report['mel']['f1-score'], 4),
        'recall_mel':    round(report['mel']['recall'], 4),
        'precision_mel': round(report['mel']['precision'], 4),
        'f1_nv':         round(report['nv']['f1-score'], 4),
        'accuracy':      round(report['accuracy'], 4),
    }
    done_marker.write_text(json.dumps(metrics, indent=2))
    (run_dir / 'classification_report.json').write_text(json.dumps(report, indent=2))

    cfg = json.loads((run_dir / 'config.json').read_text())
    cfg['finished_at']  = datetime.now(timezone.utc).isoformat()
    cfg['test_metrics'] = metrics
    (run_dir / 'config.json').write_text(json.dumps(cfg, indent=2))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['nv', 'mel']).plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')
    fig.tight_layout(); fig.savefig(run_dir / 'confusion_matrix.png', dpi=150); plt.close(fig)

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC — {name}'); ax.legend()
    fig.tight_layout(); fig.savefig(run_dir / 'roc_curve.png', dpi=150); plt.close(fig)

    ckpt_path.unlink(missing_ok=True)
    pointer_file.unlink(missing_ok=True)

    print(f'\n[{name}] COMPLETADO  AUC={metrics["auc"]}  Recall_mel={metrics["recall_mel"]}  F1_mel={metrics["f1_mel"]}')
    return metrics


print('Loop de entrenamiento listo')

## Cargar splits y preparar fuentes sintéticas

In [ ]:
train_nv, train_mel, val_samples, test_samples = load_splits()

n_2x = len(train_mel)  # 801 — N de control para escenarios aislados

# Fuentes disponibles
i2i_paths         = sorted((SYNTH_ROOT / 'img2img').glob('*.jpg'))
gan_paths         = sorted(GAN_64_DIR.glob('*.png'))    if GAN_64_DIR.exists()     else []
gfinal_paths      = sorted(GAN_FINAL_DIR.glob('*.png')) if GAN_FINAL_DIR.exists()  else []
lora_paths        = sorted(LORA_DIR.glob('*.jpg'))      if LORA_DIR.exists()       else []
derm_s040_paths   = sorted(DERM_S040_DIR.glob('*.jpg')) if DERM_S040_DIR.exists()  else []
derm_s005_paths   = sorted(DERM_S005_DIR.glob('*.jpg')) if DERM_S005_DIR.exists()  else []

# TI filtradas (puede tardar ~5 min la primera vez)
TI_FILTER_CACHE = EXP_ROOT / 'ti_filtered_paths.json'
ti_filtered_paths = get_filtered_ti(TI_FILTER_CACHE)

print(f'\nFuentes disponibles:')
print(f'  img2img:      {len(i2i_paths):5d}  (usando {n_2x} en _2x, {len(i2i_paths)} en img2img_all)')
print(f'  TI filtradas: {len(ti_filtered_paths):5d}  (usando {n_2x} en ti_filtered_2x)')
print(f'  GAN-64 old:   {len(gan_paths):5d}  (usando {min(n_2x, len(gan_paths))} en gan64_2x)')
print(f'  GAN-final:    {len(gfinal_paths):5d}  (usando {min(n_2x, len(gfinal_paths))} en gan_final_2x)')
print(f'  LoRA:         {len(lora_paths):5d}  (usando {min(n_2x, len(lora_paths))} en lora_2x)')
print(f'  Derm-s040:    {len(derm_s040_paths):5d}  (usando {min(n_2x, len(derm_s040_paths))} en derm_s040_2x)')
print(f'  Derm-s005:    {len(derm_s005_paths):5d}  (usando {min(n_2x, len(derm_s005_paths))} en derm_s005_2x)')

## Escenario 1 — `img2img_2x`
Real + 801 img2img (mismo N que `real_2x`, fuente aislada).

In [ ]:
results_img2img_2x = run_scenario(
    name          = 'img2img_2x',
    train_samples = train_nv + train_mel + synth_samples_from(i2i_paths, n=n_2x),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 2 — `ti_filtered_2x`
Real + 801 TI filtradas (mismo N, solo imágenes que pasan el filtro de calidad).

In [ ]:
results_ti_filtered = run_scenario(
    name          = 'ti_filtered_2x',
    train_samples = train_nv + train_mel + synth_samples_from(ti_filtered_paths, n=n_2x),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 3 — `gan64_2x`
Real + 801 imágenes WGAN-GP 64×64 (upscaleadas a 224×224 por el DataLoader).
Valida si la baja resolución original limita el aporte.

In [ ]:
if len(gan_paths) < 10:
    print('GAN-64 no disponible — omitiendo escenario. Sube gan_generated.zip a Drive.')
    results_gan64 = None
else:
    results_gan64 = run_scenario(
        name          = 'gan64_2x',
        train_samples = train_nv + train_mel + synth_samples_from(gan_paths, n=n_2x),
        val_samples   = val_samples,
        test_samples  = test_samples,
    )

## Escenario 4 — `img2img_all`
Real + todas las 2403 img2img disponibles.

In [ ]:
results_img2img_all = run_scenario(
    name          = 'img2img_all',
    train_samples = train_nv + train_mel + synth_samples_from(i2i_paths),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 5 — `img2img_only`
Reemplaza las imágenes reales de melanoma por 801 img2img. Nevus sigue siendo real.

In [ ]:
results_img2img_only = run_scenario(
    name          = 'img2img_only',
    train_samples = train_nv + synth_samples_from(i2i_paths, n=n_2x),
    val_samples   = val_samples,
    test_samples  = test_samples,
)

## Escenario 6 — `gan_final_2x`
Real + 801 GAN-final (5000 imágenes 64×64 guardadas con PIL). Compara directamente con `gan64_2x` (old, 1000 imgs).


In [ ]:
if len(gfinal_paths) < 10:
    print('GAN-final no disponible — omitiendo escenario. Sube gan_final.zip a Drive.')
    results_gan_final = None
else:
    results_gan_final = run_scenario(
        name          = 'gan_final_2x',
        train_samples = train_nv + train_mel + synth_samples_from(gfinal_paths, n=n_2x),
        val_samples   = val_samples,
        test_samples  = test_samples,
    )


## Escenario 7 — `lora_2x`
Real + 801 imágenes DreamBooth-LoRA (rank=32, 6000 steps). Compara directamente con `ti_filtered_2x` — misma base SD 1.5, diferente técnica de fine-tuning.


In [ ]:
if len(lora_paths) < 10:
    print('LoRA no disponible — omitiendo escenario. Sube lora.zip a Drive.')
    results_lora = None
else:
    results_lora = run_scenario(
        name          = 'lora_2x',
        train_samples = train_nv + train_mel + synth_samples_from(lora_paths, n=n_2x),
        val_samples   = val_samples,
        test_samples  = test_samples,
    )


## Escenario 8 — `derm_s040_2x`
Real + 801 Derm-T2IM con strength=0.40 (variaciones clínicas diversas).

Hipótesis: Derm-T2IM fue entrenado sobre HAM10000 + ISIC — mismo dominio que el test set — por lo que el distributional shift debería ser menor que con LoRA o GAN.

In [ ]:
if len(derm_s040_paths) < 10:
    print('Derm-s040 no disponible — omitiendo escenario.')
    results_derm_s040 = None
else:
    results_derm_s040 = run_scenario(
        name          = 'derm_s040_2x',
        train_samples = train_nv + train_mel + synth_samples_from(derm_s040_paths, n=n_2x),
        val_samples   = val_samples,
        test_samples  = test_samples,
    )

## Escenario 9 — `derm_s005_2x`
Real + 801 Derm-T2IM con strength=0.05 (perturbaciones casi imperceptibles).

Hipótesis: cambios mínimos preservan la apariencia clínica exacta del original → augmentation con el menor shift posible.

In [ ]:
if len(derm_s005_paths) < 10:
    print('Derm-s005 no disponible — omitiendo escenario.')
    results_derm_s005 = None
else:
    results_derm_s005 = run_scenario(
        name          = 'derm_s005_2x',
        train_samples = train_nv + train_mel + synth_samples_from(derm_s005_paths, n=n_2x),
        val_samples   = val_samples,
        test_samples  = test_samples,
    )

## Comparación completa — todos los experimentos
Incluye los 4 escenarios del notebook anterior y los 5 nuevos.

In [ ]:
ALL_SCENARIOS = [
    # Experimentos anteriores (fuentes mezcladas)
    'real_only', 'real_balanced', 'real_2x', 'synthetic_only',
    # Experimentos fuentes aisladas — ronda 1
    'img2img_2x', 'ti_filtered_2x', 'gan64_2x', 'img2img_all', 'img2img_only',
    # Ronda 2 — nuevas fuentes
    'gan_final_2x', 'lora_2x',
    # Ronda 3 — Derm-T2IM (modelo dermoscopy-specific)
    'derm_s040_2x', 'derm_s005_2x',
]

all_results = {}
for sc in ALL_SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m   = json.loads((run_dir / 'test_metrics.json').read_text())
        cfg = json.loads((run_dir / 'config.json').read_text()) if (run_dir / 'config.json').exists() else {}
        all_results[sc] = {'metrics': m, 'config': cfg, 'run_dir': str(run_dir)}
    else:
        print(f'  {sc}: no completado — omitido')

if all_results:
    rows = []
    for sc, data in all_results.items():
        m, cfg = data['metrics'], data['config']
        rows.append({
            'Escenario':     sc,
            'Train mel':     cfg.get('train_mel', '?'),
            'AUC':           m['auc'],
            'Recall mel':    m['recall_mel'],
            'F1 mel':        m['f1_mel'],
            'Precision mel': m['precision_mel'],
            'Accuracy':      m['accuracy'],
        })
    df = pd.DataFrame(rows).set_index('Escenario')
    print(df.to_string())
    (EXP_ROOT / 'comparison_isolated.json').write_text(
        json.dumps({sc: d['metrics'] for sc, d in all_results.items()}, indent=2)
    )
    print(f'\nComparación guardada en {EXP_ROOT}/comparison_isolated.json')

In [ ]:
if len(all_results) >= 2:
    sc_list   = list(all_results.keys())
    metrics_k = [('auc', 'AUC'), ('recall_mel', 'Recall Melanoma'), ('f1_mel', 'F1 Melanoma')]

    old_sc  = {'real_only', 'real_balanced', 'real_2x', 'synthetic_only'}
    derm_sc = {'derm_s040_2x', 'derm_s005_2x'}
    def get_color(sc):
        if sc in old_sc:  return '#aec6e8'
        if sc in derm_sc: return '#8FBC8F'
        return '#4C72B0'
    colors = [get_color(sc) for sc in sc_list]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, (mk, title) in zip(axes, metrics_k):
        vals  = [all_results[sc]['metrics'][mk] for sc in sc_list]
        bars  = ax.bar([s.replace('_', '\n') for s in sc_list], vals, color=colors)
        ax.set_ylim(0, 1.08)
        ax.set_title(title, fontsize=11)
        ax.tick_params(axis='x', labelsize=7)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)
        if 'real_only' in all_results:
            baseline = all_results['real_only']['metrics'][mk]
            ax.axhline(baseline, color='red', linestyle='--', linewidth=0.8, label='real_only')
            ax.legend(fontsize=8)

    from matplotlib.patches import Patch
    fig.legend(handles=[
        Patch(facecolor='#aec6e8', label='Experimentos anteriores (fuentes mezcladas)'),
        Patch(facecolor='#4C72B0', label='Fuentes aisladas — img2img / TI / GAN / LoRA'),
        Patch(facecolor='#8FBC8F', label='Fuentes aisladas — Derm-T2IM'),
    ], loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.08))
    fig.suptitle('HAM10000 — Comparación completa de escenarios', fontsize=12)
    fig.tight_layout()
    out = EXP_ROOT / 'comparison_isolated_plot.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot guardado en {out}')